# Check Counterfactual Data

In [2]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B", device="cuda")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


In [3]:
# Original sentence
original_text = "hotel kurang nyaman . [A] [O] [S] [A] hotel [O] kurang nyaman [S] negative"

# Candidate counterfactuals
candidate_texts = [
    "pelayanan sangat baik . [A] [O] [S] [A] pelayanan [O] sangat baik [S] positive",
    "kamar cukup bersih . [A] [O] [S] [A] kamar [O] cukup bersih [S] positive",
    "ac terasa dingin . [A] [O] [S] [A] ac [O] terasa dingin [S] positive",
    "tempat sangat tenang . [A] [O] [S] [A] tempat [O] sangat tenang [S] positive",
    "wifi sangat cepat . [A] [O] [S] [A] wifi [O] sangat cepat [S] positive",
    "kasur sangat empuk . [A] [O] [S] [A] kasur [O] sangat empuk [S] positive",
    "resepsionis ramah sekali . [A] [O] [S] [A] resepsionis [O] ramah sekali [S] positive",
    "suasana sangat nyaman . [A] [O] [S] [A] suasana [O] sangat nyaman [S] positive",
    "fasilitas cukup lengkap . [A] [O] [S] [A] fasilitas [O] cukup lengkap [S] positive",
    "parkiran sangat luas . [A] [O] [S] [A] parkiran [O] sangat luas [S] positive",
    "toilet bersih banget . [A] [O] [S] [A] toilet [O] bersih banget [S] positive",
    "sarapan sangat enak . [A] [O] [S] [A] sarapan [O] sangat enak [S] positive",
    "lokasi sangat strategis . [A] [O] [S] [A] lokasi [O] sangat strategis [S] positive",
    "menu cukup variatif . [A] [O] [S] [A] menu [O] cukup variatif [S] positive",
    "interior sangat menarik . [A] [O] [S] [A] interior [O] sangat menarik [S] positive",
    "lampu cukup terang . [A] [O] [S] [A] lampu [O] cukup terang [S] positive",
    "kebersihan terjaga baik . [A] [O] [S] [A] kebersihan [O] terjaga baik [S] positive",
    "ac cukup dingin . [A] [O] [S] [A] ac [O] cukup dingin [S] positive",
    "handuk bersih dan wangi . [A] [O] [S] [A] handuk [O] bersih dan wangi [S] positive",
    "check-in sangat cepat . [A] [O] [S] [A] check-in [O] sangat cepat [S] positive"
]




original_tokens = model.to_str_tokens(original_text, prepend_bos=False)
original_len = len(original_tokens)

# Extract original prompt
original_prompt = original_text.split("[A] [O] [S]")[0].strip()
original_prompt_len = len(model.to_str_tokens(original_prompt, prepend_bos=False))

# Extract original aspect and opinion
original_split = original_text.split("[A] [O] [S]")[-1]
original_aspect = original_split.split("[A]")[1].split("[O]")[0][:-1]
original_opinion = original_split.split("[O]")[1].split("[S]")[0][:-1]
original_aspect_tokens = model.to_str_tokens(original_aspect, prepend_bos=False)
original_opinion_tokens = model.to_str_tokens(original_opinion, prepend_bos=False)
original_aspect_len = len(original_aspect_tokens)
original_opinion_len = len(original_opinion_tokens)

# Check each candidate
for candidate in candidate_texts:
    candidate_tokens = model.to_str_tokens(candidate, prepend_bos=False)
    candidate_len = len(candidate_tokens)

    # Extract prompt
    candidate_prompt = candidate.split("[A] [O] [S]")[0].strip()
    candidate_prompt_len = len(model.to_str_tokens(candidate_prompt, prepend_bos=False))
    prompt_match = candidate_prompt_len == original_prompt_len

    # Extract aspect and opinion from second half
    candidate_split = candidate.split("[A] [O] [S]")[-1].strip()
    try:
        aspect = candidate_split.split("[A]")[1].split("[O]")[0][:-1]
        opinion = candidate_split.split("[O]")[1].split("[S]")[0][:-1]
    except:
        aspect = "?"
        opinion = "?"

    aspect_tokens = model.to_str_tokens(aspect, prepend_bos=False)
    opinion_tokens = model.to_str_tokens(opinion, prepend_bos=False)
    aspect_len = len(aspect_tokens)
    opinion_len = len(opinion_tokens)

    aspect_match = aspect_len == original_aspect_len
    opinion_match = opinion_len == original_opinion_len

    print("Original Sentence:")
    print(f"Original prompt     : '{original_prompt}' → Length: {original_prompt_len}")
    print(f"Original aspect     : '{original_aspect}' → Tokens: {original_aspect_tokens} → Length: {original_aspect_len}")
    print(f"Original opinion    : '{original_opinion}' → Tokens: {original_opinion_tokens} → Length: {original_opinion_len}")
    print(f"Original total toks : {original_len}")
    print("-" * 80)

    print(f"Candidate: {candidate}")
    print(f"Prompt  : '{candidate_prompt}' → Length: {candidate_prompt_len} (match: {prompt_match})")
    print(f"Aspect  : '{aspect}' → Tokens: {aspect_tokens} → Length: {aspect_len} (match: {aspect_match})")
    print(f"Opinion : '{opinion}' → Tokens: {opinion_tokens} → Length: {opinion_len} (match: {opinion_match})")
    print(f"Total tokens: {candidate_len} (original: {original_len})")
    print(f"Match: {prompt_match and aspect_match and opinion_match}")
    print("=" * 100)
    print("\n")

Original Sentence:
Original prompt     : 'hotel kurang nyaman .' → Length: 6
Original aspect     : ' hotel' → Tokens: [' hotel'] → Length: 1
Original opinion    : ' kurang nyaman' → Tokens: [' kur', 'ang', ' ny', 'aman'] → Length: 4
Original total toks : 30
--------------------------------------------------------------------------------
Candidate: pelayanan sangat baik . [A] [O] [S] [A] pelayanan [O] sangat baik [S] positive
Prompt  : 'pelayanan sangat baik .' → Length: 6 (match: True)
Aspect  : ' pelayanan' → Tokens: [' p', 'elay', 'anan'] → Length: 3 (match: False)
Opinion : ' sangat baik' → Tokens: [' sangat', ' baik'] → Length: 2 (match: False)
Total tokens: 30 (original: 30)
Match: False


Original Sentence:
Original prompt     : 'hotel kurang nyaman .' → Length: 6
Original aspect     : ' hotel' → Tokens: [' hotel'] → Length: 1
Original opinion    : ' kurang nyaman' → Tokens: [' kur', 'ang', ' ny', 'aman'] → Length: 4
Original total toks : 30
--------------------------------------